# 00 — Generate Test Data (SCAFFOLDING — not part of the production pipeline)

**What this notebook is:** a stand-in for a real ERP extract. It runs `generate_data.py` and packages the output as a single ZIP you download and hand to notebook 01.

**Why a file hand-off, not a shared Colab session:** two separately-opened notebooks in Colab are not guaranteed to share the same runtime, even if opened close together — that ambiguity caused a FileNotFoundError the first time these were split. A downloaded file is unambiguous: notebook 01 either has it or it doesn't, and this also mirrors production more honestly — a real raw extract *would* arrive as a file, not as shared memory between two processes.

**Output of this notebook:** one file, `ibp_raw_data.zip`, downloaded to your computer, containing `data_primary/raw/` and `data_control/raw/`. Upload it into notebook 01 when prompted there.

## Setup — clone the repo fresh

In [ ]:
import subprocess, os

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir('/content/ibp-tradeoff')
print('Working directory:', os.getcwd())


## Generate the primary dataset
Expect: net sales EUR 196,904,149 · conversion cost 0.127 PASS · roll-forward 0.000000 PASS

In [ ]:
sh('python generate_data.py')
sh('mv data data_primary')


## Generate the control dataset (low censoring)
Same script, swapped config, no code edits.
Expect: net sales EUR 199,190,133 · conversion cost 0.124 PASS

In [ ]:
sh('cp config/assumptions.yaml config/_primary_backup.yaml')
sh('cp config/assumptions_lowcensoring.yaml config/assumptions.yaml')
sh('python generate_data.py')
sh('cp config/_primary_backup.yaml config/assumptions.yaml')
sh('mv data data_control')

print('folders now:', sorted(d for d in os.listdir('.') if os.path.isdir(d)))


## Check: datasets differ only in censoring rate
Expect: `data_primary` ~6.16% · `data_control` ~1.57%

In [ ]:
import pandas as pd
for tag in ['data_primary', 'data_control']:
    a = pd.read_csv(f'{tag}/raw/plant_system_A.csv')
    b = pd.read_csv(f'{tag}/raw/plant_system_B.csv')
    cen = (pd.concat([a.STOCK_CLOSE, b.stock_eom]) <= 0).mean()
    print(f'{tag}: censoring {cen:.2%}')


## Package and download
Zips only the `raw/` folders (not `_truth/` — ground truth stays out of this hand-off deliberately, since it's only used later for scoring, in this same Claude Project, never for building the estimator). Downloads one file: **`ibp_raw_data.zip`**.

In [ ]:
sh('zip -r /content/ibp_raw_data.zip data_primary/raw data_control/raw')

from google.colab import files
files.download('/content/ibp_raw_data.zip')

print()
print('Downloaded ibp_raw_data.zip.')
print('Next: open 01_ingest_and_clean.ipynb (a fresh runtime is fine) and upload this file when prompted.')
